## Example 1: Basic PDF generation with the canvas

In [ ]:
# Example 1: Basic PDF generation with the canvas
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter, A4
from reportlab.lib.units import inch
from reportlab.lib import colors
import io

print("=== Basic PDF Generation ===")

# Page sizes are (width, height) in points (1/72 inch)
assert letter == (612.0, 792.0), letter
assert abs(A4[0] - 595.276) < 0.01 and abs(A4[1] - 841.890) < 0.01, A4
assert inch == 72.0
for name, size in [('Letter', letter), ('A4', A4)]:
    print(f"{name} page size: {size[0]:.1f} x {size[1]:.1f} points ({size[0]/inch:.2f} x {size[1]/inch:.2f} in)")

# Draw on an in-memory canvas and save it
pdf_buffer = io.BytesIO()
c = canvas.Canvas(pdf_buffer, pagesize=letter)
c.setTitle("Canvas example")
c.setFont('Helvetica', 12)
c.drawString(72, 720, 'Hello, ReportLab!')
c.setFont('Helvetica-Bold', 14)
c.drawString(72, 700, 'This is bold text')
c.setStrokeColor(colors.red)
c.line(72, 690, 200, 690)
c.setFillColor(colors.blue)
c.rect(72, 650, 100, 30, fill=1)
c.circle(300, 665, 15)
c.showPage()
c.save()

pdf = pdf_buffer.getvalue()
assert pdf.startswith(b'%PDF-'), pdf[:16]
assert pdf.rstrip().endswith(b'%%EOF'), pdf[-16:]
assert len(pdf) > 500, len(pdf)
assert b'/Count 1' in pdf  # one page in the page tree
print(f"PDF written: {len(pdf)} bytes, header {pdf[:8]!r}")

## Example 2: Document styles and text measurement

In [ ]:
# Example 2: Document styles and text measurement
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib import colors
from reportlab.pdfbase.pdfmetrics import stringWidth

print("Testing reportlab styles:")

# Get sample stylesheet
styles = getSampleStyleSheet()
for name in ['Normal', 'BodyText', 'Heading1', 'Title']:
    assert name in styles, name
    style = styles[name]
    print(f"✓ {name}: font={style.fontName}, size={style.fontSize}")
assert (styles['Normal'].fontName, styles['Normal'].fontSize) == ('Helvetica', 10)
assert styles['Heading1'].fontName == 'Helvetica-Bold'

# A custom style inherits what it does not override from its parent
custom = ParagraphStyle('Custom', parent=styles['Normal'], fontSize=14, textColor=colors.blue, alignment=TA_CENTER)
assert custom.fontName == 'Helvetica' and custom.fontSize == 14 and custom.alignment == TA_CENTER
print(f"✓ Custom style: {custom.name}, size={custom.fontSize}, color={custom.textColor}")

# Text width comes from the standard Helvetica font metrics: 27.336 points at 12 pt
width = stringWidth('Hello', 'Helvetica', 12)
assert abs(width - 27.336) < 0.01, width
assert abs(stringWidth('Hello', 'Helvetica', 24) - 2 * width) < 1e-9
print(f"✓ stringWidth('Hello', Helvetica 12) = {width:.3f} pt")

## Example 3: Graphics and shapes rendered to PDF

In [ ]:
# Example 3: Graphics and shapes with reportlab.graphics
from reportlab.graphics.shapes import Drawing, Circle, Rect, Line, String
from reportlab.graphics import renderPDF
from reportlab.lib import colors

print("=== Graphics and Shapes ===")

drawing = Drawing(400, 300)
for x, radius, color in [(100, 30, colors.red), (200, 25, colors.blue), (300, 35, colors.green)]:
    drawing.add(Circle(x, 200, radius, fillColor=color, strokeColor=colors.black))
for x, width, height, color in [(50, 80, 40, colors.yellow), (150, 100, 50, colors.orange), (270, 60, 60, colors.purple)]:
    drawing.add(Rect(x, 100, width, height, fillColor=color, strokeColor=colors.black))
drawing.add(Line(50, 50, 350, 50, strokeColor=colors.black, strokeWidth=2))
drawing.add(String(200, 280, 'Graphics Demo', fontSize=16, textAnchor='middle'))

assert (drawing.width, drawing.height) == (400, 300)
assert len(drawing.contents) == 8, len(drawing.contents)
assert [type(shape).__name__ for shape in drawing.contents].count('Circle') == 3

# Colors convert between representations
assert colors.red.hexval() == '0xff0000'
assert colors.Color(1, 0.5, 0).hexval() == '0xff7f00'

# Render the drawing to a one-page PDF in memory
pdf = renderPDF.drawToString(drawing)
assert pdf.startswith(b'%PDF-') and len(pdf) > 500, (pdf[:16], len(pdf))
print(f"Drawing with {len(drawing.contents)} shapes rendered to {len(pdf)} PDF bytes")

## Example 4: Charts in a drawing

In [ ]:
# Example 4: Charts in a drawing
from reportlab.graphics.charts.piecharts import Pie
from reportlab.graphics.charts.barcharts import VerticalBarChart
from reportlab.graphics.shapes import Drawing
from reportlab.graphics import renderPDF
from reportlab.lib import colors

print("Testing reportlab charts:")

drawing = Drawing(400, 200)

pie = Pie()
pie.x, pie.y, pie.width, pie.height = 30, 50, 100, 100
pie.data = [35, 25, 20, 15, 5]
pie.labels = ['A', 'B', 'C', 'D', 'E']
for i, color in enumerate([colors.red, colors.blue, colors.green, colors.yellow, colors.orange]):
    pie.slices[i].fillColor = color
drawing.add(pie)

bar = VerticalBarChart()
bar.x, bar.y, bar.width, bar.height = 200, 50, 180, 125
bar.data = [[450, 520, 610, 580]]
bar.categoryAxis.categoryNames = ['Q1', 'Q2', 'Q3', 'Q4']
bar.valueAxis.valueMin = 0
drawing.add(bar)

assert sum(pie.data) == 100 and len(pie.labels) == len(pie.data)
assert pie.slices[0].fillColor == colors.red
assert max(bar.data[0]) == 610

pdf = renderPDF.drawToString(drawing)
assert pdf.startswith(b'%PDF-') and len(pdf) > 500, (pdf[:16], len(pdf))
print(f"✓ Pie chart: {len(pie.data)} slices; bar chart: {len(bar.data[0])} bars; PDF {len(pdf)} bytes")

## Example 5: Multi-page Platypus document with a page template

In [ ]:
# Example 5: Multi-page Platypus document with a page template
from reportlab.platypus import BaseDocTemplate, PageTemplate, Frame, PageBreak
from reportlab.platypus import Paragraph, Table, TableStyle, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from reportlab.lib import colors
import io

print("=== Platypus document ===")

pages_drawn = []


class CustomDocTemplate(BaseDocTemplate):
    def __init__(self, filename, **kwargs):
        super().__init__(filename, **kwargs)
        frame = Frame(self.leftMargin, self.bottomMargin, self.width, self.height, id='normal')
        self.addPageTemplates([PageTemplate(id='main', frames=frame, onPage=self._header_footer)])

    def _header_footer(self, canvas, doc):
        """Draw a header and a footer on every page."""
        canvas.saveState()
        canvas.setFont('Helvetica-Bold', 12)
        canvas.drawString(72, doc.pagesize[1] - 50, "Document Title")
        canvas.setFont('Helvetica', 9)
        canvas.drawString(72, 30, f"Page {doc.page}")
        canvas.restoreState()
        pages_drawn.append(doc.page)


styles = getSampleStyleSheet()
table_data = [
    ['Product', 'Q1', 'Q2', 'Q3', 'Q4', 'Total'],
    ['Product A', '100', '120', '110', '130', '460'],
    ['Product B', '80', '90', '95', '100', '365'],
    ['Total', '180', '210', '205', '230', '825'],
]
table = Table(table_data)
table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
    ('ALIGN', (1, 1), (-1, -1), 'RIGHT'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('GRID', (0, 0), (-1, -1), 1, colors.black),
]))
story = [
    Paragraph("Quarterly report", styles['Title']),
    Paragraph("Sales by product and quarter.", styles['Normal']),
    Spacer(1, 0.25 * inch),
    table,
    PageBreak(),
    Paragraph("Appendix", styles['Heading1']),
]

buffer = io.BytesIO()
doc = CustomDocTemplate(buffer, pagesize=letter, title='Advanced ReportLab Document', author='ReportLab Example')
doc.build(story)
pdf = buffer.getvalue()

# The totals row adds up, the template ran once per page, and the PDF has two pages
for column in range(1, 6):
    assert int(table_data[3][column]) == int(table_data[1][column]) + int(table_data[2][column])
assert pages_drawn == [1, 2], pages_drawn
assert pdf.startswith(b'%PDF-') and len(pdf) > 1000, (pdf[:16], len(pdf))
assert b'/Count 2' in pdf
print(f"✓ Built a {pages_drawn[-1]}-page PDF ({len(pdf)} bytes) with header and footer on each page")